![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)







# Import in apple verion
### Check Critical Package Version
✅ JAX: 0.6.2
✅ MuJoCo: 3.3.6
✅ Brax: 0.13.0
✅ Flax: 0.10.7

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from learning.notebooks.apple_mujoco_setup import *
import jax
import jax.numpy as jnp
import numpy as np
import mediapy
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.acme import running_statistics
from flax import serialization
from mujoco_playground import registry
import wandb
import os
import matplotlib.pyplot as plt




Failed to import warp: No module named 'warp'
Failed to import mujoco.mjx.third_party.mujoco_warp as mujoco_warp: No module named 'warp'
Detected macOS: arm64
Using MUJOCO_GL=glfw for macOS
Forcing JAX to run on CPU backend
Mujoco installation and rendering backend OK
JAX: 0.6.2
MuJoCo: 3.3.6
Brax: 0.13.0
Flax: 0.10.7

Checking media packages...
✓ ffmpeg available
✓ mediapy available
[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5), CpuDevice(id=6), CpuDevice(id=7)]
JAX device count: 8


In [2]:
user = "weissma6-zhaw-school-of-engineering"
project = "UR10_pick_ppo"
run_id = "lift_512Envs_20260124_164012_5264"  # Replace with your run

# --- Create output folder ---
os.makedirs("evaluation/downloaded_policies", exist_ok=True)

# --- Download artifact ---
api = wandb.Api()
run = api.run(f"{user}/{project}/{run_id}")

# Find the policy artifact
artifacts = run.logged_artifacts()
policy_artifact = None
for artifact in artifacts:
    if artifact.type == "model":
        policy_artifact = artifact
        break

if policy_artifact is None:
    raise ValueError(f"No model artifact found for run {run_id}")

print(f"Found artifact: {policy_artifact.name}")

# Download to local folder
artifact_dir = policy_artifact.download(root="downloaded_policies")
print(f"Downloaded to: {artifact_dir}")

# Load the parameters
params_path = os.path.join(artifact_dir, "params.msgpack")
with open(params_path, "rb") as f:
    params_bytes = f.read()

print(f"✓ Loaded params ({len(params_bytes) / 1024:.1f} KB)")

wandb: Currently logged in as: weissma6 (weissma6-zhaw-school-of-engineering) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Found artifact: policy_parameters_lift_512Envs_20260124_164012_5264:v0


wandb:   2 of 2 files downloaded.  


Downloaded to: downloaded_policies
✓ Loaded params (1116.7 KB)


In [3]:
# Check top-level structure
import msgpack

with open(params_path, "rb") as f:
    raw = msgpack.unpackb(f.read(), raw=False, strict_map_key=False)

print(f"Top-level type: {type(raw)}")
print(f"Number of elements: {len(raw)}")
print(f"Keys/indices: {list(raw.keys()) if isinstance(raw, dict) else range(len(raw))}")

# Show structure of each top-level element
for i, item in enumerate(raw) if isinstance(raw, (list, tuple)) else raw.items():
    if isinstance(raw, dict):
        i, item = i, raw[i]
    print(f"\n=== Element {i} ===")
    if isinstance(item, dict):
        print(f"  Keys: {list(item.keys())}")
    else:
        print(f"  Type: {type(item)}")

Top-level type: <class 'dict'>
Number of elements: 3
Keys/indices: ['0', '1', '2']

=== Element 0 ===
  Keys: ['mean', 'std', 'count', 'summed_variance']

=== Element 1 ===
  Keys: ['params']

=== Element 2 ===
  Keys: ['params']


In [4]:
import jax
import jax.numpy as jnp
import numpy as np
import mediapy
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.acme import running_statistics
from flax import serialization
from mujoco_playground import registry

# --- Config ---
env_name = "UR10PickCube"
episode_length = 150
seed = 42
video_path = "evaluation/graphs/"
video_tag = "rollout_video.mp4"
camera_kwargs = {"camera": "side_130", "width": 800, "height": 600}
os.makedirs(video_path, exist_ok=True)

# --- Load environment ---
env = registry.load(env_name)
jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)

# --- Rebuild network ---
obs_size = env.observation_size
action_size = env.action_size
print(f"Obs size: {obs_size}, Action size: {action_size}")

normalize = running_statistics.normalize
ppo_network = ppo_networks.make_ppo_networks(
    observation_size=obs_size,
    action_size=action_size,
    preprocess_observations_fn=normalize,
)

# --- Build template matching saved structure ---
rng = jax.random.PRNGKey(0)

dummy_normalizer_params = running_statistics.init_state(
    jax.ShapeDtypeStruct((obs_size,), jnp.float32)
)
dummy_policy_params = ppo_network.policy_network.init(rng)
dummy_value_params = ppo_network.value_network.init(rng)

params_template = {
    "0": dummy_normalizer_params,
    "1": dummy_policy_params,
    "2": dummy_value_params,
}

# --- Deserialize ---
params = serialization.from_bytes(params_template, params_bytes)
normalizer_params = params["0"]
policy_params = params["1"]
value_params = params["2"]
print("✓ Params restored")

# --- Run rollout ---
rng = jax.random.PRNGKey(seed)
rng, reset_rng = jax.random.split(rng)
state = jit_reset(reset_rng)

rollout = [state]
total_reward = 0.0

for step in range(episode_length):
    rng, act_rng = jax.random.split(rng)
    
    # Normalize and apply
    obs_norm = running_statistics.normalize(state.obs, normalizer_params)
    raw_output = ppo_network.policy_network.apply(normalizer_params, policy_params, obs_norm)
    
    # Network outputs [mean, log_std] concatenated -> take first half (mean)
    action = raw_output[:action_size]
    
    state = jit_step(state, action)
    rollout.append(state)
    total_reward += float(state.reward)

print(f"✓ Rollout complete | Total reward: {total_reward:.2f}")

# --- Render video ---
frames = env.render(rollout, **camera_kwargs)
frames = np.asarray(frames).astype(np.uint8)

fps = int(1.0 / env.dt)

full_video_path = os.path.join(video_path, video_tag)
mediapy.write_video(full_video_path, frames, fps=fps)
print(f"✓ Video saved to {full_video_path}")

# Display in notebook
mediapy.show_video(frames, fps=fps)

✓ Using keyframe: 'low_home'
  Initial qpos size: 15
  Robot joints: [ 0.   -1.7   2.25 -2.15 -1.5  -1.5   0.    0.  ]
Available sensors:
  - left_finger_pad_floor_found
  - right_finger_pad_floor_found
  - hand_capsule_floor_found
  - box_hand_found
  - left_finger_box_contact
  - right_finger_box_contact
  - tcp_position
  - box_position
Obs size: 63, Action size: 7
✓ Params restored
[RESET] robot_qpos=[ 0.   -1.7   2.25 -2.15 -1.5  -1.5   0.    0.  ] | box_qpos=[0.7  0.2  0.03] | ctrl=[ 0.   -1.7   2.25 -2.15 -1.5  -1.5   0.  ] | ctrl-qpos-err=0.0


/Users/matthiasweiss/miniconda3/envs/mujoco/lib/python3.10/site-packages/jax/_src/interpreters/xla.py:119: RuntimeWarning: overflow encountered in cast
  return np.asarray(x, dtypes.canonicalize_dtype(x.dtype))


✓ Rollout complete | Total reward: 42.29


100%|██████████| 151/151 [00:01<00:00, 98.69it/s] 


✓ Video saved to evaluation/graphs/rollout_video.mp4


This browser does not support the video tag.
